# ASEAN PTCST-v2 — build data and run forecast development

This notebook does not use Vietnam data. It reuses the existing `asean_v1_country_runs` folder/ZIP in Drive, creates a separate V2 workspace, and leaves V1 results untouched. 2024–2025 are development evidence; do not present them as an untouched final test.

In [ ]:
# Cell 1 — Mount Drive and clone the current code
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
from pathlib import Path
import subprocess, sys, shutil, zipfile
REPO = Path('/content/kltn')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/maiphuowng205/kltn.git', str(REPO)], check=True)
print('Commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO/'requirements-colab.txt')], check=True)

In [ ]:
# Cell 2 — Reuse ASEAN V1 data already on Drive; no re-upload is needed
DRIVE = Path('/content/drive/MyDrive')
LOCAL_COUNTRY_ROOT = Path('/content/asean_v1_country_runs')
folder = DRIVE/'kltn'/'asean_v1_country_runs'
zip_path = DRIVE/'kltn'/'asean_v1_country_runs.zip'
if folder.is_dir():
    source = folder
    if LOCAL_COUNTRY_ROOT.exists(): shutil.rmtree(LOCAL_COUNTRY_ROOT)
    shutil.copytree(source, LOCAL_COUNTRY_ROOT)
elif zip_path.is_file():
    if LOCAL_COUNTRY_ROOT.exists(): shutil.rmtree(LOCAL_COUNTRY_ROOT)
    with zipfile.ZipFile(zip_path) as z: z.extractall('/content')
    if not LOCAL_COUNTRY_ROOT.exists(): raise FileNotFoundError('ZIP must contain asean_v1_country_runs at its top level.')
else:
    raise FileNotFoundError('Cannot find MyDrive/kltn/asean_v1_country_runs or asean_v1_country_runs.zip.')
print('V1 country source:', LOCAL_COUNTRY_ROOT)

In [ ]:
# Cell 3 — Build and validate a separate V2 dataset (high-RAM full ASEAN mode)
V1_SOURCE = Path('/content/asean_v1_source')
V2_DATA = Path('/content/asean_v2')
# These are generated local workspaces only; remove a partial prior build.
for generated in (V1_SOURCE, V2_DATA):
    if generated.exists(): shutil.rmtree(generated)
subprocess.run([sys.executable, str(REPO/'scripts/assemble_asean_v1_source_from_country_runs.py'), '--country-root', str(LOCAL_COUNTRY_ROOT), '--output-root', str(V1_SOURCE), '--load-all'], check=True)
subprocess.run([sys.executable, str(REPO/'scripts/build_asean_v2_dataset.py'), '--source-root', str(V1_SOURCE), '--output-root', str(V2_DATA), '--risk-min-history', '126', '--load-all'], check=True)
subprocess.run([sys.executable, str(REPO/'scripts/validate_asean_v2_contract.py'), '--data-root', str(V2_DATA)], check=True)
print('V2 dataset ready:', V2_DATA)

In [ ]:
# Cell 4 — Train the pooled ASEAN PTCST-v2 forecast model (five seeds)
# This is development training only. It does not claim a final 2024–2025 test.
RUN_FORECAST_TRAINING = True
V2_RUN = Path('/content/asean_v2_runs/pooled_ptcst')
if RUN_FORECAST_TRAINING:
    subprocess.run([sys.executable, str(REPO/'scripts/run_asean_v2_forecasts.py'), '--data-root', str(V2_DATA), '--run-root', str(V2_RUN), '--epochs', '100', '--seeds', '7,19,31,43,59'], check=True)
else:
    print('Training skipped. Change RUN_FORECAST_TRAINING to True when ready.')

In [ ]:
# Cell 5 — Save dataset report and forecast outputs to Drive
DRIVE_OUT = DRIVE/'kltn'/'asean_v2_development'
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
shutil.copytree(V2_DATA/'reports', DRIVE_OUT/'dataset_reports', dirs_exist_ok=True)
if V2_RUN.exists(): shutil.copytree(V2_RUN, DRIVE_OUT/'pooled_ptcst', dirs_exist_ok=True)
print('Saved to:', DRIVE_OUT)